# Morning class 27/08 — Worksheet 14 SOLUTIONS: capstone   (L06)

Every cell below was executed on the same Python the lab ships (3.13), and the
quoted output is what it actually printed.

Each question builds on the functions defined before it, so run them in order.
By Q8 you have a pipeline that fits on one line.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 14 — Capstone. Run this once.

# The same shape of feed as worksheet 07: amounts are TEXT, and some rows
# are unusable.
feed_a = [
    {"id": 101, "user": "u1", "amount": "12.50"},
    {"id": 102, "user": "u2", "amount": "8.00"},
    {"id": 103, "user": "u1", "amount": ""},
    {"id": 104, "user": "u9", "amount": "31.00"},
    {"id": 105, "user": "u2", "amount": "-4.00"},
    {"id": 106, "user": "u3", "amount": "17.25"},
    {"id": 107, "user": "u1", "amount": "n/a"},
    {"id": 108, "user": "u3", "amount": "9.99"},
]

# A second feed, arriving later, with different data and the same problems.
feed_b = [
    {"id": 201, "user": "u2", "amount": "40.00"},
    {"id": 202, "user": "u7", "amount": "5.50"},
    {"id": 203, "user": "u3", "amount": "0.00"},
    {"id": 204, "user": "u1", "amount": "oops"},
]

customers = {"u1": "north", "u2": "south", "u3": "north"}

print(len(feed_a), "rows in feed_a,", len(feed_b), "in feed_b")

PART A — One job per function

### Question 1

Parsing one field. -> `'12.50' -> 12.5`, `'-4.00' -> -4.0`, `'' -> None`, `'n/a' -> None`, `'0.00' -> 0.0`.

One job, one function, testable on five inputs in three lines. Worksheet 07
Q5 had this same logic buried inside a loop, where the only way to try it
on a new value was to edit the feed.

Note that `-4.00` **parses fine**. Parsing and validating are different
questions — "is this a number" and "is this an allowed number" — and mixing
them into one function is how you end up unable to tell a malformed field
from a negative one. Q2 does the second job.

`'0.00' -> 0.0` matters too: `0.0` is falsy, so `if parse_amount(x):` would
treat a legitimate zero as a failure. That is why Q2 tests
`value is None` and not `if not value`.

In [ ]:
def parse_amount(text):
    """Return `text` as a float, or None if it is blank or not a number."""
    if not text:
        return None
    if not text.replace(".", "", 1).replace("-", "", 1).isdigit():
        return None
    return float(text)

for sample in ["12.50", "-4.00", "", "n/a", "0.00"]:
    print(repr(sample), "->", parse_amount(sample))

### Question 2

Row by row. -> five rows as `(amount, None)`; `103 (None, 'blank amount')`, `105 (None, 'negative')`, `107 (None, 'not a number')`.

Returning `(value, reason)` gives the caller both halves of the answer: what
the row is worth, and — if it is worth nothing — why. A function that
returned only `True`/`False` would force the caller to work the reason out
again.

The order of the checks is load-bearing, exactly as in worksheet 07 Q5. A
blank has to be distinguished from a non-number **before** you conclude it
is a non-number, because `parse_amount` returns `None` for both.

And `check_row` calls `parse_amount` rather than repeating it. Each
function knows about one layer.

In [ ]:
def check_row(row):
    """Return (amount, None) if the row is usable, else (None, reason)."""
    value = parse_amount(row["amount"])
    if value is None:
        if not row["amount"]:
            return None, "blank amount"
        return None, "not a number"
    if value < 0:
        return None, "negative"
    return value, None

for row in feed_a:
    print(row["id"], check_row(row))

### Question 3

Splitting the feed. -> `5 clean, 3 rejected`; `[(103, 'blank amount'), (105, 'negative'), (107, 'not a number')]`; `8 rows in -- unchanged: True`.

Five clean and three rejected out of eight — and `feed_a` is untouched.
That last line is the check worth building the habit around: `validate`
builds new dictionaries rather than editing the caller's rows, so you can
run it twice, or run it and still have the original to compare against.
Worksheet 11 Q5 is what the other choice looks like.

Returning **two** lists rather than one is the design decision here. A
function that returned only `clean` would be throwing away the information
nobody looks for until 3am.

In [ ]:
def validate(rows):
    """Split `rows` into (clean, rejects). Does not modify `rows`."""
    clean = []
    rejects = []
    for row in rows:
        value, reason = check_row(row)
        if reason is not None:
            rejects.append((row["id"], reason))
            continue
        clean.append({"id": row["id"], "user": row["user"], "amount": value})
    return clean, rejects

clean_a, rejects_a = validate(feed_a)
print(len(clean_a), "clean,", len(rejects_a), "rejected")
print(rejects_a)
print(len(feed_a), "rows in -- unchanged:", len(feed_a) == 8)

PART B — Functions general enough to reuse

### Question 4

A join that does not know what it is joining. -> `4 joined, 1 orphaned: [104]`, then `{'id': 101, 'user': 'u1', 'amount': 12.5, 'region': 'north', 'tier': 'core'}`.

The word `customers` does not appear inside `join`, and neither does
`region`. Both arrive as arguments, which is why the same function joined a
second, completely different lookup — region to tier — with no changes at
all.

That is the difference between **extracting** a function and **designing**
one. Extracting worksheet 07's join would have given you
`join_customers(rows)`, which is tidier but no more reusable than the loop
was.

`dict(row)` makes a shallow copy so the caller's rows do not sprout new
keys. And `orphans` comes back separately for the same reason as Q3's
rejects: row 104 is still `31.00` of revenue that did not make it into any
region.

In [ ]:
def join(rows, lookup, key_field, new_field):
    """Add `new_field` from `lookup`. Return (joined, orphan ids)."""
    joined = []
    orphans = []
    for row in rows:
        key = row[key_field]
        if key not in lookup:
            orphans.append(row["id"])
            continue
        enriched = dict(row)              # a copy -- do not touch the caller's row
        enriched[new_field] = lookup[key]
        joined.append(enriched)
    return joined, orphans

joined_a, orphans_a = join(clean_a, customers, "user", "region")
print(len(joined_a), "joined,", len(orphans_a), "orphaned:", orphans_a)

tiers = {"north": "core", "south": "growth"}
tiered_a, tier_orphans = join(joined_a, tiers, "region", "tier")
print(tiered_a[0])

### Question 5

One function, three reports. -> by region `{'north': 39.74, 'south': 8.0}`; by user `{'u1': 12.5, 'u2': 8.0, 'u3': 27.240000000000002}`; by tier `{'core': 39.74, 'growth': 8.0}`.

Three group-bys from one nine-line function, because the field names are
**arguments** rather than something written into the loop. Worksheet 07
needed a separate loop for each.

That is the real test of a reusable function: not "can I call it twice" but
"can I call it for a question I had not thought of when I wrote it". The
tier report did not exist until Q4 invented it.

`u3: 27.240000000000002` is float addition again — 17.25 + 9.99. Correct to
fifteen places and not something to print in a report without formatting.

In [ ]:
def summarise(rows, group_field, value_field):
    """Total `value_field` per distinct `group_field`."""
    totals = {}
    for row in rows:
        group = row[group_field]
        if group in totals:
            totals[group] = totals[group] + row[value_field]
        else:
            totals[group] = row[value_field]
    return totals

print("by region:", summarise(tiered_a, "region", "amount"))
print("by user:  ", summarise(tiered_a, "user", "amount"))
print("by tier:  ", summarise(tiered_a, "tier", "amount"))

PART C — Passing behaviour, not just data

### Question 6

Passing a rule in. -> `2 over 10`, `3 in the north`, `3 with an even id`.

`keep_if` contains a loop and an `if`, and has no opinion about what the
`if` is testing. The caller supplies that as a **function**, which is
worksheet 12's `key=` idea from the other side: there, you passed a
function that computed a sort value; here you pass one that makes a
decision.

This is what worksheet 09 Q5 was for. A function is a value, so it can be
an argument, and once it can be an argument you can write loops that do not
know what they are for.

Three different questions, none of which required touching `keep_if`. A
fourth — rows from a particular user, rows above the median — is one more
lambda.

In [ ]:
def keep_if(rows, rule):
    """Return the rows for which `rule(row)` is true."""
    kept = []
    for row in rows:
        if rule(row):
            kept.append(row)
    return kept

print(len(keep_if(tiered_a, lambda r: r["amount"] > 10)), "over 10")
print(len(keep_if(tiered_a, lambda r: r["region"] == "north")), "in the north")
print(len(keep_if(tiered_a, lambda r: r["id"] % 2 == 0)), "with an even id")

### Question 7

The pipeline, with a switch. -> default: `{'rows_in': 8, 'rejected': 3, 'orphaned': 1, 'below_min': 0, 'kept': 4, 'total': 47.74, 'by_region': {'north': 39.74, 'south': 8.0}}`; with `min_amount=10.0`: `'below_min': 2, 'kept': 2, 'total': 29.75, 'by_region': {'north': 29.75}`.

Twelve lines, and every one of them is a call to something already written
and already tested. That is the shape a pipeline should end up in.

`min_amount=0.0` as a **default** means the common case costs nothing at the
call site and the variation is one keyword away — worksheet 10 Q1. A float
default, not a list, so worksheet 10 Q4's trap does not apply here.

Look at what the report counts. `rejected`, `orphaned`, `below_min` and
`kept` are four separate buckets, deliberately, because "8 in, 4 out" is
not a diagnosis. Raising the threshold moved two rows from `kept` to
`below_min` and took `south` out of the report entirely — the region did
not vanish, its only row was worth 8.00.

In [ ]:
def run(rows, lookup, min_amount=0.0):
    """Validate, join and summarise `rows`. Return a report dict."""
    clean, rejects = validate(rows)
    joined, orphans = join(clean, lookup, "user", "region")
    kept = keep_if(joined, lambda r: r["amount"] >= min_amount)
    return {
        "rows_in": len(rows),
        "rejected": len(rejects),
        "orphaned": len(orphans),
        "below_min": len(joined) - len(kept),
        "kept": len(kept),
        "total": sum(r["amount"] for r in kept),
        "by_region": summarise(kept, "region", "amount"),
    }

print(run(feed_a, customers))
print(run(feed_a, customers, min_amount=10.0))

### Question 8

The second feed. -> `feed_b: {'rows_in': 4, 'rejected': 1, 'orphaned': 1, 'below_min': 0, 'kept': 2, 'total': 40.0, 'by_region': {'south': 40.0, 'north': 0.0}}`; both: `rows_in: 12, kept: 6, total: 87.74`.

**Three lines.** No new validation, no new join, no new summariser — the
second feed had the same shape, so the same pipeline handled it. That is
the entire claim of slides 43–58, and this is what it looks like when it is
true.

Combining the feeds was `feed_a + feed_b`, and the numbers add up: 8 + 4
rows, 3 + 1 rejects, 47.74 + 40.00 = 87.74.

One figure in `feed_b` deserves a second look: `'north': 0.0`. That is not
a missing region — it is row 203, a genuine `0.00` from u3, correctly parsed
(Q1) and correctly totalled. A region worth nothing and a region with no
rows print very differently here, and only because `parse_amount` was
careful about zero.

In [ ]:
report_a = run(feed_a, customers)
report_b = run(feed_b, customers)

print("feed_a:", report_a)
print("feed_b:", report_b)

report_both = run(feed_a + feed_b, customers)
print("both:  ", report_both)

### Question 9

Reconciliation. -> all three reports `reconciles: True | totals agree: True`.

`reconcile` is worksheet 07 Q11 as a function you can call on every run
instead of a paragraph you read once. Every input row ends in exactly one
of four buckets, and if that ever stops being true, rows are disappearing
somewhere nobody is looking.

Writing it as a function is the point: a check you have to remember to do by
eye is a check that will not be done. This one costs one line per run.

`check_total` uses `abs(parts - total) < 0.01` rather than `==` for the
reason worksheet 08 Q10 demonstrated — summing by region and summing
overall round differently, and an `==` reconciliation fails on correct
data. It passes here at `True`; on the real superstore totals it would not
have.

In [ ]:
def reconcile(report):
    """True if every input row is accounted for in exactly one bucket."""
    accounted = (report["rejected"] + report["orphaned"]
                 + report["below_min"] + report["kept"])
    return accounted == report["rows_in"]

def check_total(report):
    """True if the per-region totals add back up to the grand total."""
    parts = sum(report["by_region"].values())
    return abs(parts - report["total"]) < 0.01

for name, report in [("a", report_a), ("b", report_b), ("both", report_both)]:
    print(name, "reconciles:", reconcile(report),
          "| totals agree:", check_total(report))

PART D — The function that is right and useless

### Question 10

Averages. -> `{'north': 13.246666666666668, 'south': 8.0}`; rows behind each: `{'north': 3, 'south': 1}`; empty input `-> {}`.

Both averages are correct. **`south: 8.0` is the average of one row.**

Nothing in the returned dictionary says so. Printed in a report, `north
13.25 / south 8.00` invites the reader to compare two regions, and one of
them is a single transaction from a single customer. Add a second south row
and the figure could double.

That is the failure this whole series has been circling, and it survives
being wrapped in a well-named function — arguably it is *worse* here,
because `average_by_region(rows)` looks authoritative in a way that a loop
does not. **A function is not a guarantee of meaning.** If a number needs a
denominator to be honest, the function has to return the denominator.

The empty case returns `{}` rather than raising, which is the right
behaviour and also the quiet one: worksheet 07 Q8 again — an empty report
and a broken pipeline look identical.

So: make it return counts alongside the means, and let the caller decide
what is worth showing. Then this sheet's toolkit is finished.

In [ ]:
def average_by_region(rows):
    """Mean amount per region. Says nothing about how many rows each is from."""
    totals = summarise(rows, "region", "amount")
    counts = {}
    for row in rows:
        counts[row["region"]] = counts.get(row["region"], 0) + 1
    return {region: totals[region] / counts[region] for region in totals}

kept_a = keep_if(join(validate(feed_a)[0], customers, "user", "region")[0],
                 lambda r: True)

print(average_by_region(kept_a))

counts = {}
for row in kept_a:
    counts[row["region"]] = counts.get(row["region"], 0) + 1
print("rows behind each average:", counts)

print("empty input ->", average_by_region([]))